# Prepare the historical weather dataset

This notebook downloads daily ERA5 weather data from the Open-Meteo Historical Weather API, combines the six city files, checks data quality and saves one analysis-ready CSV.

Existing raw CSV files are reused. This avoids unnecessary API requests and rate-limit errors when the notebook is run again.


## 1. Project setup

Configuration is stored in `scripts/config.py`. Download, cleaning and validation functions are stored in `scripts/data.py`.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import pandas as pd

from scripts.config import CITIES, DAILY_VARIABLES, END_DATE, START_DATE
from scripts.data import (
    clean_weather_data,
    download_all_cities,
    load_raw_weather,
    save_processed_data,
    validate_weather_data,
)

pd.set_option("display.max_columns", None)


## 2. Download configuration

The same ERA5 model and daily variables are used for every city and year. This provides a consistent source for comparisons over time.


In [3]:
RAW_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_FILE = PROJECT_ROOT / "data" / "processed" / "weather_daily_1940_2025.csv"
CITY_SLUGS = [city["slug"] for city in CITIES]

city_table = pd.DataFrame(CITIES)
display(city_table)
print(f"Period: {START_DATE} to {END_DATE}")
print(f"Daily variables: {', '.join(DAILY_VARIABLES)}")


,name,slug,latitude,longitude
0,Milan,milan,45.4642,9.1900
1,Turin,turin,45.0703,7.6869
2,Rome,rome,41.9028,12.4964
3,Bari,bari,41.1171,16.8719
4,Palermo,palermo,38.1157,13.3615
5,Cagliari,cagliari,39.2238,9.1217


Period: 1940-01-01 to 2025-12-31
Daily variables: temperature_2m_mean, temperature_2m_max, temperature_2m_min, precipitation_sum


## 3. Download or reuse raw city files

`download_all_cities()` makes one request per city only when its raw CSV does not already exist. The function raises a readable error for failed requests and waits before retrying HTTP 429 rate-limit responses.

Set `overwrite=True` only when the source files must be downloaded again.


In [4]:
raw_files = download_all_cities(
    CITIES,
    RAW_DIRECTORY,
    overwrite=False,
)

for path in raw_files:
    print(Path(PROJECT_ROOT.name) / path.relative_to(PROJECT_ROOT))


Using existing file: milan_19400101_20251231_era5.csv
Using existing file: turin_19400101_20251231_era5.csv
Using existing file: rome_19400101_20251231_era5.csv
Using existing file: bari_19400101_20251231_era5.csv
Using existing file: palermo_19400101_20251231_era5.csv
Using existing file: cagliari_19400101_20251231_era5.csv
Weather-Report/data/raw/milan_19400101_20251231_era5.csv
Weather-Report/data/raw/turin_19400101_20251231_era5.csv
Weather-Report/data/raw/rome_19400101_20251231_era5.csv
Weather-Report/data/raw/bari_19400101_20251231_era5.csv
Weather-Report/data/raw/palermo_19400101_20251231_era5.csv
Weather-Report/data/raw/cagliari_19400101_20251231_era5.csv


## 4. Load and clean

The six CSV files are combined with `pandas.concat()`. Dates are parsed as datetimes, weather columns are converted to numeric values and rows are sorted by city and date.


In [5]:
raw_weather = load_raw_weather(raw_files)
weather = clean_weather_data(raw_weather)

print(f"Raw rows: {len(raw_weather):,}")
print(f"Cleaned rows: {len(weather):,}")
display(weather.head())
display(weather.dtypes.to_frame("dtype"))


Raw rows: 188,472
Cleaned rows: 188,472


,city,city_slug,requested_latitude,requested_longitude,grid_latitude,grid_longitude,grid_elevation,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum
0,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-01,1.6,3.9,-1.6,NaN
1,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-02,2.6,4.9,1.2,1.3
2,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-03,1.5,4.4,-0.6,0.0
3,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-04,6.6,9.0,2.3,7.6
4,Bari,bari,41.1171,16.8719,41.0,16.75,9.0,1940-01-05,9.4,12.0,7.5,11.1


,dtype
city,str
city_slug,str
requested_latitude,float64
requested_longitude,float64
grid_latitude,float64
grid_longitude,float64
grid_elevation,float64
date,datetime64[us]
temperature_2m_mean,float64
temperature_2m_max,float64


## 5. Validate data quality

The validation checks:

- all six expected cities;
- complete daily coverage from 1940 through 2025;
- no duplicate city-date rows;
- no missing temperature values;
- minimum ≤ mean ≤ maximum temperature;
- no negative precipitation;
- only the documented missing precipitation value on 1 January 1940.

Missing precipitation is never replaced with zero.


In [6]:
quality_summary = validate_weather_data(
    weather,
    CITY_SLUGS,
    start_date=START_DATE,
    end_date=END_DATE,
)
quality_summary


,city_slug,rows,first_date,last_date,missing_precipitation
0,milan,31412,1940-01-01,2025-12-31,1
1,turin,31412,1940-01-01,2025-12-31,1
2,rome,31412,1940-01-01,2025-12-31,1
3,bari,31412,1940-01-01,2025-12-31,1
4,palermo,31412,1940-01-01,2025-12-31,1
5,cagliari,31412,1940-01-01,2025-12-31,1


## 6. Save the processed dataset

The validated table is saved as one tidy CSV. Each row represents one city on one date and each weather measurement has its own column.


In [7]:
saved_path = save_processed_data(weather, PROCESSED_FILE)
print(f"Saved {len(weather):,} rows")
print(Path(PROJECT_ROOT.name) / saved_path.relative_to(PROJECT_ROOT))


Saved 188,472 rows
Weather-Report/data/processed/weather_daily_1940_2025.csv


## Result

The processed dataset is ready for the climate analysis notebook. It contains real API data, transparent quality checks and the original missing precipitation values.
